# AI-Powered Construction-Site Vehicle Classification System
### End-to-End Pipeline: EfficientNetB0 Transfer Learning on ConstructionXC7C Dataset

This notebook provides the complete 17-step pipeline from dataset ingestion to evaluation and custom image inference.

## 1. Install Dependencies

In [ ]:
!pip install -q tensorflow keras scikit-learn matplotlib seaborn pillow pyyaml requests huggingface_hub

## 2. Check GPU & Environment

In [ ]:
import tensorflow as tf
print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU Acceleration: ACTIVE ({len(gpus)} GPU found)')
    for gpu in gpus:
        print(' ', gpu)
else:
    print('GPU Acceleration: None (Running on CPU)')

## 3. Clone Repository & Setup Configuration

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project files are in workspace
if not Path('config/config.yaml').exists():
    print('Please ensure this notebook is executed within the project repository.')
sys.path.insert(0, os.path.abspath('.'))

## 4. Download ConstructionXC7C Dataset

In [ ]:
from src.dataset.download import setup_dataset
# Download validation and test sets (2,281 real annotated vehicle images)
setup_dataset(config_path='config/config.yaml', splits=['test', 'valid'])

## 5. Inspect Dataset & Generate Health Report

In [ ]:
from src.dataset.inspect import inspect_dataset
report = inspect_dataset(config_path='config/config.yaml', classes_config_path='config/classes.yaml')
print(f"Total Images: {report['total_images_found']} | Annotations: {report['total_annotations_found']}")
print(f"Annotation Format: {report['annotation_format_detected']}")
print("Annotations per Class:", report['annotations_per_class'])

## 6. Parse Annotations & Extract Vehicle Bounding Box Crops

In [ ]:
from src.dataset.create_crops import generate_vehicle_crops
meta_path, quality_path = generate_vehicle_crops(
    config_path='config/config.yaml',
    classes_config_path='config/classes.yaml'
)
print(f"Generated crops recorded in: {meta_path}")

## 7. Visualize Vehicle Classes

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

classes = ['bulldozer', 'dump_truck', 'excavator', 'grader', 'loader', 'mixer_truck', 'mobile_crane', 'roller']
plt.figure(figsize=(16, 8))
for idx, cls_name in enumerate(classes):
    sample_crops = glob.glob(f'data/crops/{cls_name}/*.jpg')
    if sample_crops:
        plt.subplot(2, 4, idx + 1)
        plt.imshow(Image.open(sample_crops[0]))
        plt.title(cls_name.replace('_', ' ').title(), fontsize=12, fontweight='bold')
        plt.axis('off')
plt.tight_layout()
plt.show()

## 8. Split Dataset (Leakage-Free Partitioning)

In [ ]:
from src.dataset.split import split_dataset
split_info = split_dataset(config_path='config/config.yaml')
print("Split completed successfully with zero source image overlap:")
print(f"Train Crops: {split_info['train_crops']} | Val Crops: {split_info['val_crops']} | Test Crops: {split_info['test_crops']}")

## 9. Build Optimized tf.data Pipelines

In [ ]:
from src.preprocessing.preprocessing import build_data_pipelines
train_ds, val_ds, test_ds, canonical_classes = build_data_pipelines(
    config_path='config/config.yaml',
    classes_config_path='config/classes.yaml'
)
print(f"Ready with {len(canonical_classes)} canonical classes:", canonical_classes)

## 10. Build EfficientNetB0 Model

In [ ]:
from src.models.efficientnet import build_efficientnetb0, freeze_backbone, count_parameters
model = build_efficientnetb0(num_classes=len(canonical_classes), image_size=224, dropout_rate=0.3)
freeze_backbone(model)
tot, trn, ntr = count_parameters(model)
print(f"Total Params: {tot:,} | Trainable (Head): {trn:,} | Non-trainable: {ntr:,}")

## 11. Stage 1: Feature Extraction Training

In [ ]:
from src.training.train import run_stage1_training
model_stage1, history1 = run_stage1_training(
    config_path='config/config.yaml',
    classes_config_path='config/classes.yaml',
    epochs_override=12
)

## 12. Stage 2: Fine-Tuning EfficientNetB0

In [ ]:
from src.training.fine_tune import run_stage2_fine_tuning
final_model, history2 = run_stage2_fine_tuning(
    config_path='config/config.yaml',
    classes_config_path='config/classes.yaml',
    epochs_override=10
)

## 13. Evaluate on Untouched Test Set

In [ ]:
from src.evaluation.evaluate import evaluate_model_on_test
eval_results = evaluate_model_on_test(config_path='config/config.yaml', classes_config_path='config/classes.yaml')
print(f"Test Accuracy : {eval_results['test_accuracy'] * 100:.2f}%")
print(f"Macro F1-Score: {eval_results['macro_metrics']['f1_score']:.4f}")

## 14. Confusion Matrix & Training Curves

In [ ]:
from src.evaluation.confusion_matrix import generate_all_matrices_and_curves
generate_all_matrices_and_curves(config_path='config/config.yaml', classes_config_path='config/classes.yaml')

# Display confusion matrix
from IPython.display import Image as IPImage, display
display(IPImage('reports/confusion_matrix.png'))
display(IPImage('reports/training_accuracy.png'))

## 15. Error Analysis & Confident Errors

In [ ]:
from src.evaluation.error_analysis import perform_error_analysis
perform_error_analysis(config_path='config/config.yaml', classes_config_path='config/classes.yaml')
if Path('reports/misclassified_grid.png').exists():
    display(IPImage('reports/misclassified_grid.png'))

## 16. Verify Exported Model & Classes

In [ ]:
!ls -lh models/

## 17. Single Image Prediction & Top-K Telemetry

In [ ]:
import glob
from src.inference.predict import VehiclePredictor, format_cli_output

test_images = glob.glob('data/test/*/*.jpg')
if test_images:
    sample_img = test_images[0]
    print(f"Evaluating sample image: {sample_img}")
    predictor = VehiclePredictor()
    res = predictor.predict(sample_img, top_k=3, threshold=0.50)
    print(format_cli_output(res))
    display(IPImage(sample_img))